# Feature Engineering part 2 - Redundant Feature Elimination

In [1]:
PROJECT_DIR = ".."
DATA_DIR = f"{PROJECT_DIR}/data"
ARTIFACTS_DIR = PROJECT_DIR

## Elimination by Feature Clustering

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import home_credit_risk.data_util as data_util

SEED = 42

def eliminate_redundant_features(df: pd.DataFrame, n_clusters: int) -> pd.DataFrame:
    scaler = StandardScaler(with_mean=False)
    X = df.copy()
    print(f"Clustering {len(X.columns)} features into {n_clusters} clusters...")
    X.fillna(0, inplace=True)
    X = scaler.fit_transform(X).T
    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED)
    kmeans.fit(X)
    representatives = []
    for cluster_id in range(n_clusters):
        cluster_idx = np.where(kmeans.labels_ == cluster_id)[0]
        cluster_features = X[cluster_idx]
        centroid = kmeans.cluster_centers_[cluster_id]
        distances = np.linalg.norm(cluster_features - centroid, axis=1)
        representatives.append(cluster_idx[np.argmin(distances)])

    df_reduced = df.iloc[:, representatives].copy()   # Copy to avoid warnings
    print(f"Dataset reduced to shape {df_reduced.shape}")
    return df_reduced

## Shrink Training Dataset

In [4]:
n_features_to_keep = 1500
train_df = pd.read_parquet(f"{DATA_DIR}/generated_features_all_train.parquet")
train_df.set_index(data_util.APP_ID_COL, inplace=True)
train_df = eliminate_redundant_features(train_df, n_clusters=n_features_to_keep)

Clustering 6298 features into 1500 clusters...
Dataset reduced to shape (307511, 1500)


In [5]:
file_base_path = f"{DATA_DIR}/generated_features_{n_features_to_keep}"
train_df.to_csv(f"{file_base_path}_train.csv")
kept_features = train_df.columns.tolist()
with open(f"{file_base_path}.txt", "w") as f:
    f.write("\n".join(kept_features))

## Shrink Test Dataset

In [6]:
test_df = pd.read_parquet(f"{DATA_DIR}/generated_features_all_test.parquet")
test_df = test_df[[data_util.APP_ID_COL, *kept_features]]
test_df.to_csv(f"{file_base_path}_test.csv")